# LiveCodeBench: extracción local de prompts

Requiere `modeling/data/raw/livecodebench/test*.jsonl`. Si no existen, informa la ausencia y no genera resultados ficticios. No descarga datos. Cada salida incluye solo `prompt`; deduplica dentro de cada nivel.

In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "modeling/src/data/extract_prompts.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA = ROOT / "modeling/data"


In [2]:
import json
from collections import defaultdict
from modeling.src.data.extract_prompts import export_prompts
RAW = DATA / "raw/livecodebench"
files = sorted(RAW.glob("test*.jsonl"))
if not files:
    print(f"SIN DATOS: no hay test*.jsonl en {RAW}. Extracción pendiente de proporcionar esos archivos.")
else:
    levels = defaultdict(list)
    skipped = 0
    for path in files:
        with path.open(encoding="utf-8") as stream:
            for number, line in enumerate(stream, 1):
                if not line.strip():
                    continue
                try:
                    row = json.loads(line)
                except json.JSONDecodeError as error:
                    raise ValueError(f"JSON inválido: {path}:{number}") from error
                level = str(row.get("difficulty", "")).lower()
                if level not in {"easy", "medium", "hard"}:
                    skipped += 1
                    continue
                levels[level].append(row.get("question_content"))
    for level, prompts in sorted(levels.items()):
        print(level, export_prompts(prompts, DATA / "processed/por_nivel_livecodebench" / f"{level}.jsonl"))
    print(f"Descartados por dificultad no reconocida: {skipped}")


SIN DATOS: no hay test*.jsonl en /Users/emiliogarcialopez/projectGitHub/deep_learning_Project/prompt-difficulty-predictor/modeling/data/raw/livecodebench. Extracción pendiente de proporcionar esos archivos.
